# Analysis Dashboard: Autonomous Driving System Performance

This notebook provides an interactive dashboard for exploring the performance of our multi-task perception and driving policy system.

## Contents
1. [Overview](#overview)
2. [Detection Performance](#detection)
3. [Segmentation Performance](#segmentation)
4. [Policy Performance](#policy)
5. [System-Level Analysis](#system)
6. [Ablation Studies](#ablation)
7. [Failure Modes](#failures)

In [ ]:
# Import required libraries
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully")

<a id='overview'></a>
## 1. Performance Overview

Load and display key metrics from all benchmark runs.

In [ ]:
# Load metrics
results_dir = Path("../results")

try:
    with open(results_dir / "metrics.json", 'r') as f:
        metrics = json.load(f)
    print("✓ Metrics loaded successfully")
except FileNotFoundError:
    print("⚠️ Metrics file not found. Run 'python scripts/benchmark_all.py' first.")
    metrics = {}

# Display hardware info
if 'hardware' in metrics:
    print("\n=== Hardware Configuration ===")
    for key, value in metrics['hardware'].items():
        print(f"{key}: {value}")

In [ ]:
# Create overview summary table
overview_data = []

if 'detection' in metrics:
    overview_data.append({
        'Module': 'Object Detection',
        'Primary Metric': 'mAP@0.5',
        'Value': f"{metrics['detection'].get('mAP@0.5', 0):.3f}",
        'FPS': f"{metrics['detection'].get('fps', 0):.1f}"
    })

if 'segmentation' in metrics and 'lane' in metrics['segmentation']:
    overview_data.append({
        'Module': 'Lane Segmentation',
        'Primary Metric': 'IoU',
        'Value': f"{metrics['segmentation']['lane'].get('iou', 0):.3f}",
        'FPS': f"{metrics['segmentation']['lane'].get('fps', 0):.1f}"
    })

if 'segmentation' in metrics and 'drivable_area' in metrics['segmentation']:
    overview_data.append({
        'Module': 'Drivable Area Seg',
        'Primary Metric': 'IoU',
        'Value': f"{metrics['segmentation']['drivable_area'].get('iou', 0):.3f}",
        'FPS': f"{metrics['segmentation']['drivable_area'].get('fps', 0):.1f}"
    })

if 'policy' in metrics:
    overview_data.append({
        'Module': 'Driving Policy',
        'Primary Metric': 'RMSE',
        'Value': f"{metrics['policy'].get('rmse', 0):.4f}",
        'FPS': f"{metrics['policy'].get('fps', 0):.1f}"
    })

if 'system' in metrics:
    overview_data.append({
        'Module': 'Full System',
        'Primary Metric': 'Total Latency',
        'Value': f"{metrics['system'].get('total_latency_ms', 0):.1f} ms",
        'FPS': f"{metrics['system'].get('fps', 0):.1f}"
    })

df_overview = pd.DataFrame(overview_data)
print("\n=== Performance Overview ===")
print(df_overview.to_string(index=False))

<a id='detection'></a>
## 2. Object Detection Performance

Detailed analysis of YOLOv8 detection performance.

In [ ]:
if 'detection' in metrics:
    det = metrics['detection']
    
    # Create metrics bar chart
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy metrics
    acc_metrics = {
        'mAP@0.5': det.get('mAP@0.5', 0),
        'mAP@0.75': det.get('mAP@0.75', 0),
        'Precision': det.get('precision', 0),
        'Recall': det.get('recall', 0),
        'F1': det.get('f1', 0)
    }
    
    ax1.bar(acc_metrics.keys(), acc_metrics.values(), color=sns.color_palette("husl", len(acc_metrics)))
    ax1.set_ylabel('Score')
    ax1.set_title('Detection Accuracy Metrics')
    ax1.set_ylim([0, 1.0])
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # Per-class performance
    if 'per_class' in det:
        classes = list(det['per_class'].keys())
        aps = [det['per_class'][c].get('ap', 0) for c in classes]
        
        ax2.barh(classes, aps, color=sns.color_palette("viridis", len(classes)))
        ax2.set_xlabel('Average Precision')
        ax2.set_title('Per-Class Performance')
        ax2.set_xlim([0, 1.0])
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nInference Time: {det.get('inference_time_ms', 0):.1f} ms")
    print(f"FPS: {det.get('fps', 0):.1f}")
else:
    print("⚠️ Detection metrics not available")

<a id='segmentation'></a>
## 3. Segmentation Performance

Analysis of lane and drivable area segmentation.

In [ ]:
if 'segmentation' in metrics:
    seg = metrics['segmentation']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Lane segmentation
    if 'lane' in seg:
        lane_metrics = {
            'IoU': seg['lane'].get('iou', 0),
            'Dice': seg['lane'].get('dice', 0),
            'Pixel Acc': seg['lane'].get('pixel_acc', 0)
        }
        
        ax1.bar(lane_metrics.keys(), lane_metrics.values(), color='steelblue', alpha=0.7)
        ax1.set_ylabel('Score')
        ax1.set_title('Lane Segmentation Metrics')
        ax1.set_ylim([0, 1.0])
        
        for i, (k, v) in enumerate(lane_metrics.items()):
            ax1.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    
    # Drivable area segmentation
    if 'drivable_area' in seg:
        drivable_metrics = {
            'IoU': seg['drivable_area'].get('iou', 0),
            'Dice': seg['drivable_area'].get('dice', 0),
            'Pixel Acc': seg['drivable_area'].get('pixel_acc', 0)
        }
        
        ax2.bar(drivable_metrics.keys(), drivable_metrics.values(), color='forestgreen', alpha=0.7)
        ax2.set_ylabel('Score')
        ax2.set_title('Drivable Area Segmentation Metrics')
        ax2.set_ylim([0, 1.0])
        
        for i, (k, v) in enumerate(drivable_metrics.items()):
            ax2.text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Segmentation metrics not available")

<a id='policy'></a>
## 4. Driving Policy Performance

Analysis of steering prediction performance.

In [ ]:
if 'policy' in metrics:
    pol = metrics['policy']
    
    # Create metrics display
    fig, ax = plt.subplots(figsize=(8, 5))
    
    policy_metrics = {
        'RMSE': pol.get('rmse', 0),
        'MAE': pol.get('mae', 0),
        'Correlation': pol.get('correlation', 0)
    }
    
    colors = ['coral', 'salmon', 'lightblue']
    bars = ax.bar(policy_metrics.keys(), policy_metrics.values(), color=colors, alpha=0.7)
    ax.set_ylabel('Value')
    ax.set_title('Policy Prediction Metrics')
    
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}',
                ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nInference Time: {pol.get('inference_time_ms', 0):.1f} ms")
    print(f"FPS: {pol.get('fps', 0):.1f}")
    print(f"Sequence Length: {pol.get('sequence_length', 1)}")
else:
    print("⚠️ Policy metrics not available")

<a id='system'></a>
## 5. System-Level Analysis

Overall system performance and latency breakdown.

In [ ]:
if 'system' in metrics:
    sys_metrics = metrics['system']
    
    # Latency breakdown pie chart
    if 'components' in sys_metrics:
        components = sys_metrics['components']
        names = list(components.keys())
        latencies = [components[name]['latency_ms'] for name in names]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        
        # Pie chart
        colors = sns.color_palette("Set2", len(names))
        ax1.pie(latencies, labels=names, autopct='%1.1f%%', colors=colors, startangle=90)
        ax1.set_title('Latency Distribution')
        
        # Bar chart
        ax2.barh(names, latencies, color=colors)
        ax2.set_xlabel('Latency (ms)')
        ax2.set_title('Component Latencies')
        
        plt.tight_layout()
        plt.show()
    
    print(f"\n=== System Performance ===")
    print(f"Total Latency: {sys_metrics.get('total_latency_ms', 0):.1f} ms")
    print(f"System FPS: {sys_metrics.get('fps', 0):.1f}")
else:
    print("⚠️ System metrics not available")

<a id='ablation'></a>
## 6. Ablation Studies

Analysis of architectural choices and their impact.

In [ ]:
# Load ablation results
try:
    with open(results_dir / "ablations.json", 'r') as f:
        ablations = json.load(f)
    print("✓ Ablation results loaded")
    
    # Multi-task comparison
    if 'multi_task_comparison' in ablations:
        mt_comp = ablations['multi_task_comparison']
        
        configs = [k for k in mt_comp.keys() if k != 'analysis']
        latencies = [mt_comp[k]['latency_ms'] for k in configs]
        memories = [mt_comp[k]['memory_gb'] for k in configs]
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
        
        ax1.bar(configs, latencies, color=['coral', 'steelblue'])
        ax1.set_ylabel('Latency (ms)')
        ax1.set_title('Multi-Task vs Single-Task: Latency')
        plt.setp(ax1.xaxis.get_majorticklabels(), rotation=20, ha='right')
        
        ax2.bar(configs, memories, color=['coral', 'steelblue'])
        ax2.set_ylabel('Memory (GB)')
        ax2.set_title('Multi-Task vs Single-Task: Memory')
        plt.setp(ax2.xaxis.get_majorticklabels(), rotation=20, ha='right')
        
        plt.tight_layout()
        plt.show()
        
        savings = (latencies[0] - latencies[1]) / latencies[0] * 100
        print(f"\nMulti-task learning reduces latency by {savings:.1f}%")
        
except FileNotFoundError:
    print("⚠️ Ablation results not found. Run 'python scripts/run_ablations.py' first.")

<a id='failures'></a>
## 7. Failure Mode Analysis

Understanding when and why the system fails.

In [ ]:
# Load failure analysis
try:
    with open(results_dir / "failures.json", 'r') as f:
        failures = json.load(f)
    print("✓ Failure analysis loaded")
    
    # Failure distribution
    if 'failure_modes' in failures:
        modes = failures['failure_modes']
        names = list(modes.keys())
        counts = [modes[name]['affected_samples'] for name in names]
        
        # Sort by count
        sorted_data = sorted(zip(names, counts), key=lambda x: x[1], reverse=True)
        names, counts = zip(*sorted_data[:8])  # Top 8
        
        fig, ax = plt.subplots(figsize=(12, 6))
        bars = ax.barh(names, counts, color=sns.color_palette("Spectral", len(names)))
        ax.set_xlabel('Number of Affected Samples')
        ax.set_title('Top Failure Modes')
        ax.invert_yaxis()
        
        # Add percentage labels
        total = failures.get('total_samples', 100)
        for i, (bar, count) in enumerate(zip(bars, counts)):
            pct = count / total * 100
            ax.text(count + 0.5, i, f'{pct:.1f}%', va='center')
        
        plt.tight_layout()
        plt.show()
        
        print(f"\nFailure Rate: {failures.get('failure_rate', 0):.1f}%")
        print(f"Total Failed Samples: {failures.get('failed_samples', 0)} / {total}")
        
except FileNotFoundError:
    print("⚠️ Failure analysis not found. Run 'python scripts/analyze_failures.py' first.")

## Summary

This dashboard provides an interactive view of system performance. Key findings:

1. **Detection**: YOLOv8 provides real-time object detection
2. **Segmentation**: U-Net achieves good IoU for lane and drivable area
3. **Policy**: ConvLSTM temporal model improves prediction stability
4. **System**: Multi-task learning reduces latency while maintaining accuracy
5. **Failures**: Most failures occur in challenging lighting and occlusion scenarios

For detailed analysis, see:
- `docs/performance_metrics.md`
- `docs/ablation_studies.md`
- `docs/failure_analysis.md`
- `docs/profiling_report.md`